# Uber Fare Prediction

**Dataset:** `uber_trips_dataset_50k.csv`
**Target:** `fare_amount`
**Models:** Linear Regression, Random Forest

## Setup — Install and Import Libraries

In [1]:
!pip install -q joblib

import os, json, warnings, zipfile
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
sns.set_style("whitegrid")

## Create Output Folders

In [2]:
OUT = "/content/outputs"
os.makedirs(f"{OUT}/charts", exist_ok=True)
os.makedirs(f"{OUT}/models", exist_ok=True)

## Load the Dataset

Reads `uber_trips_dataset_50k.csv` directly from `/content/`. No upload prompt.

In [3]:
CSV_PATH = "/content/uber_trips_dataset_50k.csv"
assert os.path.exists(CSV_PATH), f"CSV not found at {CSV_PATH}. Upload it to /content/ first."
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')
print(df.shape)
df.head()

(50000, 14)


,trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time
0,1,8270,10683,San Francisco,37.170931,-77.586479,37.173652,-77.619934,2.97,10.71,Completed,Wallet,2023-01-01 00:00:00,2023-01-01 00:08:54.600000000
1,2,1860,44743,Boston,38.898127,-108.582977,38.937464,-108.558727,8.43,22.41,Completed,UPI,2023-01-01 00:01:00,2023-01-01 00:26:17.400000000
2,3,6390,75839,San Francisco,38.814571,-89.942603,38.821702,-89.896435,5.46,12.91,Completed,Cash,2023-01-01 00:02:00,2023-01-01 00:18:22.800000000
3,4,6191,22189,New York,37.295906,-75.328844,37.301375,-75.317488,6.61,15.70,Completed,Wallet,2023-01-01 00:03:00,2023-01-01 00:22:49.800000000
4,5,6734,61104,Seattle,38.972395,-121.482913,38.992088,-121.467904,10.50,19.15,Completed,Wallet,2023-01-01 00:04:00,2023-01-01 00:35:30.000000000


## Data Understanding

Check dtypes, missing values, duplicates, and summary statistics.

In [4]:
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df['status'].value_counts())
df.describe()

trip_id             int64
driver_id           int64
rider_id            int64
city               object
pickup_lat        float64
pickup_lng        float64
drop_lat          float64
drop_lng          float64
distance_km       float64
fare_amount       float64
status             object
payment_method     object
pickup_time        object
drop_time          object
dtype: object
trip_id           0
driver_id         0
rider_id          0
city              0
pickup_lat        0
pickup_lng        0
drop_lat          0
drop_lng          0
distance_km       0
fare_amount       0
status            0
payment_method    0
pickup_time       0
drop_time         0
dtype: int64
0
status
Completed    42540
Cancelled     4984
No-Show       2476
Name: count, dtype: int64


,trip_id,driver_id,rider_id,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,25000.500000,5493.954540,55040.994580,38.998664,-97.485552,38.998768,-97.485677,7.008070,15.976205
std,14433.901067,2601.410798,25915.467105,1.155241,14.173744,1.155696,14.173735,2.946744,6.274406
min,1.000000,1000.000000,10001.000000,37.000009,-121.999465,36.953365,-122.047124,0.000000,1.080000
25%,12500.750000,3232.750000,32539.000000,37.994428,-109.818282,37.998237,-109.817260,4.980000,11.510000
50%,25000.500000,5497.000000,55164.000000,38.992589,-97.463042,38.994190,-97.464646,6.980000,15.430000
75%,37500.250000,7746.000000,77293.500000,39.999602,-85.190954,39.998849,-85.199652,9.000000,19.910000
max,50000.000000,9998.000000,99998.000000,40.999937,-73.001961,41.047536,-72.963476,19.410000,50.670000


## Data Cleaning

Remove duplicates, cancelled/no-show trips, invalid coordinates, non-positive fares, and extreme fare outliers.

In [5]:
original = df.shape[0]
df = df.drop_duplicates()
df = df[df['status'] == 'Completed']
df = df.dropna(subset=['pickup_lat','pickup_lng','drop_lat','drop_lng','fare_amount'])
df = df[(df['pickup_lat'].between(-90, 90)) & (df['drop_lat'].between(-90, 90))]
df = df[(df['pickup_lng'].between(-180, 180)) & (df['drop_lng'].between(-180, 180))]
df = df[df['fare_amount'] > 0]
df = df[df['fare_amount'] <= df['fare_amount'].quantile(0.995)]
df = df[df['distance_km'] > 0]
print(f"{original} -> {df.shape[0]}")

50000 -> 42325


## Feature Engineering

Extract time-based features and compute Haversine trip distance from pickup/dropoff coordinates.

In [6]:
df['pickup_time'] = pd.to_datetime(df['pickup_time'], errors='coerce')
df = df.dropna(subset=['pickup_time'])
df['pickup_year']  = df['pickup_time'].dt.year
df['pickup_month'] = df['pickup_time'].dt.month
df['pickup_day']   = df['pickup_time'].dt.day
df['pickup_hour']  = df['pickup_time'].dt.hour
df['day_of_week']  = df['pickup_time'].dt.dayofweek

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['trip_distance'] = haversine(df['pickup_lat'], df['pickup_lng'],
                                df['drop_lat'], df['drop_lng'])
df.head()

,trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time,pickup_year,pickup_month,pickup_day,pickup_hour,day_of_week,trip_distance
0,1,8270,10683,San Francisco,37.170931,-77.586479,37.173652,-77.619934,2.97,10.71,Completed,Wallet,2023-01-01 00:00:00,2023-01-01 00:08:54.600000000,2023,1,1,0,6,2.979600
1,2,1860,44743,Boston,38.898127,-108.582977,38.937464,-108.558727,8.43,22.41,Completed,UPI,2023-01-01 00:01:00,2023-01-01 00:26:17.400000000,2023,1,1,0,6,4.851174
2,3,6390,75839,San Francisco,38.814571,-89.942603,38.821702,-89.896435,5.46,12.91,Completed,Cash,2023-01-01 00:02:00,2023-01-01 00:18:22.800000000,2023,1,1,0,6,4.077696
3,4,6191,22189,New York,37.295906,-75.328844,37.301375,-75.317488,6.61,15.70,Completed,Wallet,2023-01-01 00:03:00,2023-01-01 00:22:49.800000000,2023,1,1,0,6,1.174280
4,5,6734,61104,Seattle,38.972395,-121.482913,38.992088,-121.467904,10.50,19.15,Completed,Wallet,2023-01-01 00:04:00,2023-01-01 00:35:30.000000000,2023,1,1,0,6,2.545195


## Chart Saver Helper

In [7]:
def save_chart(name):
    plt.tight_layout()
    plt.savefig(f"{OUT}/charts/{name}.png", dpi=120)
    plt.close()

## EDA — Fare Distribution

In [8]:
plt.figure()
sns.histplot(df['fare_amount'], bins=50, kde=True)
plt.title('Fare Distribution')
save_chart('fare_distribution')
plt.show()

## EDA — Trip Distance Distribution

In [9]:
plt.figure()
sns.histplot(df['trip_distance'], bins=50, kde=True)
plt.title('Trip Distance Distribution')
save_chart('trip_distance_distribution')
plt.show()

## EDA — Fare vs Trip Distance

In [10]:
plt.figure()
sns.scatterplot(x='trip_distance', y='fare_amount', data=df, alpha=0.3)
plt.title('Fare vs Trip Distance')
save_chart('fare_vs_distance')
plt.show()

## EDA — Fare by Hour

In [11]:
plt.figure()
sns.boxplot(x='pickup_hour', y='fare_amount', data=df)
plt.title('Fare by Hour')
save_chart('fare_by_hour')
plt.show()

## EDA — Fare by Day of Week

In [12]:
plt.figure()
sns.boxplot(x='day_of_week', y='fare_amount', data=df)
plt.title('Fare by Day of Week (0=Mon)')
save_chart('fare_by_day_of_week')
plt.show()

## EDA — Fare by Month

In [13]:
plt.figure()
sns.boxplot(x='pickup_month', y='fare_amount', data=df)
plt.title('Fare by Month')
save_chart('fare_by_month')
plt.show()

## EDA — Correlation Matrix

In [14]:
num_cols = ['fare_amount','distance_km','trip_distance',
            'pickup_hour','day_of_week','pickup_month']
plt.figure(figsize=(8,6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
save_chart('correlation_matrix')
plt.show()

## Train / Test Split

In [15]:
FEATURES = ['distance_km','trip_distance','pickup_lat','pickup_lng',
            'drop_lat','drop_lng','pickup_hour','day_of_week','pickup_month']
TARGET = 'fare_amount'
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(33860, 9) (8465, 9)


## Evaluation Helper Function

In [16]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        'Model': name,
        'MAE':  round(mean_absolute_error(y_test, pred), 4),
        'MSE':  round(mean_squared_error(y_test, pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 4),
        'R2':   round(r2_score(y_test, pred), 4)
    }, model, pred

## Model — Linear Regression

In [17]:
lr_res, lr_model, lr_pred = evaluate("Linear Regression", LinearRegression())
print(lr_res)

{'Model': 'Linear Regression', 'MAE': 2.4654, 'MSE': 9.4143, 'RMSE': np.float64(3.0683), 'R2': 0.7428}


## Model — Random Forest Regressor

In [18]:
rf_res, rf_model, rf_pred = evaluate(
    "Random Forest",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
)
print(rf_res)

{'Model': 'Random Forest', 'MAE': 2.5127, 'MSE': 9.95, 'RMSE': np.float64(3.1544), 'R2': 0.7281}


## Model Comparison Table

In [19]:
results_df = pd.DataFrame([lr_res, rf_res])
print(results_df.to_string(index=False))
results_df.to_csv(f"{OUT}/model_comparison.csv", index=False)

            Model    MAE    MSE   RMSE     R2
Linear Regression 2.4654 9.4143 3.0683 0.7428
    Random Forest 2.5127 9.9500 3.1544 0.7281


## Save Best Model

In [20]:
best_model = rf_model if rf_res['RMSE'] < lr_res['RMSE'] else lr_model
best_name  = "Random Forest" if best_model is rf_model else "Linear Regression"
print("Best:", best_name)
joblib.dump(best_model, f"{OUT}/models/uber_fare_model.pkl")

Best: Linear Regression


['/content/outputs/models/uber_fare_model.pkl']

## Feature Importance (Random Forest)

In [21]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(8,5))
importances.plot(kind='bar')
plt.title('Feature Importance (Random Forest)')
save_chart('feature_importance')
plt.show()
print(importances)

distance_km      0.791768
trip_distance    0.048456
pickup_lat       0.029393
drop_lat         0.029205
pickup_hour      0.029178
drop_lng         0.025682
pickup_lng       0.025554
day_of_week      0.017544
pickup_month     0.003220
dtype: float64


## Actual vs Predicted (Random Forest)

In [22]:
plt.figure()
plt.scatter(y_test, rf_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Fare'); plt.ylabel('Predicted Fare')
plt.title('Actual vs Predicted (Random Forest)')
save_chart('actual_vs_predicted')
plt.show()

## Business Answers — Export as JSON

In [23]:
biz = {
    "most_influential_features": importances.head(3).to_dict(),
    "distance_correlation_with_fare": round(df['trip_distance'].corr(df['fare_amount']), 4),
    "avg_fare_by_hour": df.groupby('pickup_hour')['fare_amount'].mean().round(2).to_dict(),
    "avg_fare_by_day_of_week": df.groupby('day_of_week')['fare_amount'].mean().round(2).to_dict(),
    "avg_fare_by_month": df.groupby('pickup_month')['fare_amount'].mean().round(2).to_dict(),
    "best_model": best_name,
    "best_model_rmse": float(min(lr_res['RMSE'], rf_res['RMSE'])),
    "best_model_r2":   float(max(lr_res['R2'],   rf_res['R2'])),
}
with open(f"{OUT}/business_answers.json", "w") as f:
    json.dump(biz, f, indent=2)
print(json.dumps(biz, indent=2))

{
  "most_influential_features": {
    "distance_km": 0.7917682479964608,
    "trip_distance": 0.048455997598926624,
    "pickup_lat": 0.02939339779308391
  },
  "distance_correlation_with_fare": -0.0002,
  "avg_fare_by_hour": {
    "0": 15.61,
    "1": 15.83,
    "2": 15.7,
    "3": 16.05,
    "4": 15.74,
    "5": 15.86,
    "6": 16.09,
    "7": 15.74,
    "8": 16.12,
    "9": 16.02,
    "10": 15.99,
    "11": 15.85,
    "12": 15.92,
    "13": 15.97,
    "14": 15.7,
    "15": 15.7,
    "16": 15.89,
    "17": 15.73,
    "18": 16.13,
    "19": 15.81,
    "20": 15.96,
    "21": 15.82,
    "22": 15.69,
    "23": 15.96
  },
  "avg_fare_by_day_of_week": {
    "0": 16.0,
    "1": 15.93,
    "2": 15.86,
    "3": 15.79,
    "4": 15.88,
    "5": 15.76,
    "6": 15.86
  },
  "avg_fare_by_month": {
    "1": 15.87,
    "2": 15.87
  },
  "best_model": "Linear Regression",
  "best_model_rmse": 3.0683,
  "best_model_r2": 0.7428
}


## Zip All Outputs

In [24]:
zip_path = "/content/uber_fare_prediction_outputs.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for file in files:
            full = os.path.join(root, file)
            z.write(full, os.path.relpath(full, OUT))
print(zip_path)

/content/uber_fare_prediction_outputs.zip


## Download the Zip

In [25]:
from google.colab import files
files.download("/content/uber_fare_prediction_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>